# CNN Image Classification Benchmark

## 1. Project Overview
This notebook runs controlled CNN experiments for two tasks: smile binary classification and SIGNS multiclass digit classification. It compares baseline, improved, and augmented models, then summarizes metrics and plots produced by the training pipeline.

## 2. Environment Setup

In [ ]:
from pathlib import Path
import pandas as pd

from src import config
from src.data_loader import MissingDatasetError, load_happy_dataset, load_signs_dataset, dataset_summary, print_dataset_summary
from src.train import available_models, train_and_evaluate
from src.visualization import plot_benchmark_comparison

config.ensure_output_dirs()
config.set_global_determinism(config.SEED)
print(f"Project root: {config.PROJECT_ROOT}")
print(f"Outputs: {config.OUTPUT_DIR}")

## 3. Dataset Loading and Inspection

In [ ]:
datasets = {}
try:
    datasets["smile"] = load_happy_dataset()
    datasets["signs"] = load_signs_dataset()
except MissingDatasetError as exc:
    print(str(exc))
    raise SystemExit(1)

print_dataset_summary(datasets["smile"], "smile")
print_dataset_summary(datasets["signs"], "signs")

summary_df = pd.DataFrame([
    dataset_summary(datasets["smile"], "smile"),
    dataset_summary(datasets["signs"], "signs"),
])
summary_df

## 4. Baseline CNN Models
- Smile: `build_smile_baseline()`
- SIGNS: `build_signs_baseline()`

## 5. Improved CNN Models
- Smile: `build_smile_improved_cnn()` and `build_smile_augmented_cnn()`
- SIGNS: `build_signs_improved_cnn()` and `build_signs_augmented_cnn()`

## 6. Training and Evaluation

In [ ]:
benchmark_plan = {
    "smile": ["baseline", "improved_cnn", "augmented_cnn"],
    "signs": ["baseline", "improved_cnn", "augmented_cnn"],
}

results = []
for task, model_names in benchmark_plan.items():
    data = datasets[task]
    models = available_models(task)
    task_epochs = config.SMILE_EPOCHS if task == "smile" else config.SIGNS_EPOCHS
    batch_size = config.SMILE_BATCH_SIZE if task == "smile" else config.DEFAULT_BATCH_SIZE

    for model_name in model_names:
        print(f"\nRunning {task}/{model_name}")
        _, _, summary = train_and_evaluate(
            task=task,
            model_name=model_name,
            model_builder=models[model_name],
            data=data,
            epochs=task_epochs,
            batch_size=batch_size,
            seed=config.SEED,
        )
        summary["seed"] = config.SEED
        results.append(summary)

results_df = pd.DataFrame(results)
results_df

## 7. Benchmark Results

In [ ]:
results_df = results_df.sort_values(["task", "model", "seed"]).reset_index(drop=True)
results_csv = config.METRICS_DIR / "benchmark_results.csv"
results_json = config.METRICS_DIR / "benchmark_results.json"

results_df.to_csv(results_csv, index=False)
results_df.to_json(results_json, orient="records", indent=2)

print(f"Saved: {results_csv}")
print(f"Saved: {results_json}")
results_df

## 8. Visual Analysis

In [ ]:
plot_benchmark_comparison(results_df, output_path=config.FIGURES_DIR / "benchmark_test_accuracy.png")
print(f"Saved figure: {config.FIGURES_DIR / 'benchmark_test_accuracy.png'}")

## 9. Summary of Findings
This notebook executes a reproducible benchmark across baseline, improved, and augmented CNN variants for both tasks. Detailed per-run training logs, evaluation metrics, confusion matrices, and benchmark summary artifacts are saved under `outputs/metrics/` and `outputs/figures/`.